In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
import re
import nltk
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import string
from collections import Counter

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/fake_or_real_news.csv')
df2 = pd.read_csv('/content/drive/MyDrive/news.csv')

In [ ]:
df['title'] = df['title'].fillna('No title')

In [ ]:
df2['Kategorie'] = df2['Kategorie'].fillna('No Kategorie')

In [ ]:
df['text'] = df['text'].fillna('No text')

In [ ]:
df2 = df2.drop(['Art', 'Datum', 'url', 'id'], axis =1)

In [ ]:
fig = plt.figure(figsize=(10,6))
ax = sns.countplot(x='Fake',data=df2) 

abs_values = df2['Fake'].value_counts(ascending=False)
rel_values = df2['Fake'].value_counts(ascending=False, normalize=True).values * 100
lbls = [f'{p[0]} ({p[1]:.0f}%)' for p in zip(abs_values, rel_values)]

plt.show()

In [ ]:
sns.countplot(x='label',data=df) 

In [ ]:
df2['News'] = df2['Titel']

In [ ]:
df2 = df2.drop(['Titel','Body','Kategorie','Quelle', 'Body'], axis =1 )

In [ ]:
df2.head()

In [ ]:
import nltk
nltk.download('stopwords')

spec_chara = re.compile('[/(){}\[\]\|@,;]') # REMOVING SPECIAL CHARACTERS 
ext_sym = re.compile('[^0-9a-z #+_]') #EXTRA SYMBOLS
st = set(stopwords.words('english')) # USING STOPWORD FUNCTION
st2 = set(stopwords.words('german'))

In [ ]:
def clean(words): #Data cleaning function
    words = words.lower()
    words = spec_chara.sub(' ', words)
    words = ext_sym.sub('', words)
    words = ' '.join(word for word in words.split() if word not in st)
    return words

In [ ]:
def clean2(words): #Data cleaning function
    words = words.lower()
    words = spec_chara.sub(' ', words)
    words = ext_sym.sub('', words)
    words = ' '.join(word for word in words.split() if word not in st2)
    return words

In [ ]:
df['cleanNews'] = df['text'].apply(clean)
df = df.drop(['text', 'Unnamed: 0','title'], axis =1)

In [ ]:
df2['cleanNews'] = df2['News'].apply(clean2)
df2 = df2.drop(['News'], axis =1)

In [ ]:
from sklearn.preprocessing import LabelEncoder
label = LabelEncoder()
df['Fake']= label.fit_transform(df['label'])
df.drop(['label'], axis = 1)

In [ ]:
def convert(df):
    if df == 0:
        return 1
    else:
        return 0

In [ ]:
df['Fake'] = df['Fake'].apply(convert)
df = df.drop(['label'], axis =1)
df.head()

In [ ]:
df2.head()

In [ ]:
df = pd.concat([df, df2], axis=0)

In [ ]:
df.head()

In [ ]:
x = df['cleanNews']
y = df['Fake']

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=2)

In [ ]:
!pip install --upgrade gensim

In [ ]:
!pip install numpy==1.22

In [ ]:
import gensim

w2v_model = gensim.models.Word2Vec(X_train,
                                   vector_size=100,
                                   window=5,
                                   min_count=2)

w2v_model.save("w2v_model.model")

In [ ]:
words = set(w2v_model.wv.index_to_key )
X_train_vect = np.array([np.array([w2v_model.wv[i] for i in ls if i in words])
                         for ls in X_train])
X_test_vect = np.array([np.array([w2v_model.wv[i] for i in ls if i in words])
                         for ls in X_test])

In [ ]:
X_train_vect_avg = []
for v in X_train_vect:
    if v.size:
        X_train_vect_avg.append(v.mean(axis=0))
    else:
        X_train_vect_avg.append(np.zeros(100, dtype=float))
        
X_test_vect_avg = []
for v in X_test_vect:
    if v.size:
        X_test_vect_avg.append(v.mean(axis=0))
    else:
        X_test_vect_avg.append(np.zeros(100, dtype=float))

In [ ]:
log = LogisticRegression(solver='liblinear', penalty='l1')

In [ ]:
model = log.fit(X_train_vect_avg, y_train.values.ravel())

In [ ]:
y_pred = log.predict(X_test_vect_avg)

In [ ]:
print(accuracy_score(y_test,y_pred))

In [ ]:
cfm = confusion_matrix(y_test,y_pred)
sns.heatmap(cfm, annot=True, cmap='rainbow_r', fmt='g')

In [ ]:
print(classification_report(y_pred, y_test))

In [ ]:
support = SVC(kernel='sigmoid', gamma=1.0)

In [ ]:
model2 = support.fit(X_train_vect_avg, y_train.values.ravel())

In [ ]:
y_pred2 = support.predict(X_test_vect_avg)

In [ ]:
print(accuracy_score(y_test,y_pred2))

In [ ]:
cfm2 = confusion_matrix(y_test,y_pred2)
sns.heatmap(cfm2, annot=True, cmap='rainbow_r', fmt='g')

In [ ]:
print(classification_report(y_pred2, y_test))

In [ ]:
import pickle
filename = 'model.sav'
pickle.dump(model, open(filename, 'wb'))

In [ ]:
from gensim.models import Word2Vec
loaded_model = pickle.load(open('model.sav', 'rb'))
w2v_model = Word2Vec.load("w2v_model.model")

In [ ]:
!pip install pyngrok==4.1.1
!pip install flask_ngrok

In [ ]:
from flask import Flask, render_template, request, url_for, Markup, jsonify
import pickle
#import tensorflow
import numpy as np
from keras.preprocessing.text import one_hot
from keras_preprocessing.sequence import pad_sequences
import re
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from flask_ngrok import run_with_ngrok
import nltk
nltk.download('stopwords')

app = Flask(__name__)
run_with_ngrok(app)

!ngrok authtoken 2HXMnOGWrPvMDccFnaCSjaHf9Ve_2gqDRmc4dhRoJsQtzystb


loaded_model = pickle.load(open('model.sav', 'rb'))
w2v_model = Word2Vec.load("w2v_model.model")

spec_chara = re.compile('[/(){}\[\]\|@,;]') # REMOVING SPECIAL CHARACTERS 
ext_sym = re.compile('[^0-9a-z #+_]') #EXTRA SYMBOLS
st = set(stopwords.words('german')) # USING STOPWORD FUNCTION

def clean(words): #Data cleaning function
    words = words.lower()
    words = spec_chara.sub(' ', words)
    words = ext_sym.sub('', words)
    words = ' '.join(word for word in words.split() if word not in st)
    return words



@app.route('/')
def home():
 	return render_template("index.html")
  



@app.route('/newscheck')
def newscheck():
  abc = request.args.get('news')	
  word = clean(abc)
  iput_data = [abc.rstrip()]
  words = set(w2v_model.wv.index_to_key )
  X_train_vect = np.array([np.array([w2v_model.wv[i] for i in ls if i in words]) for ls in iput_data])
  X_train_vect_avg = []
  for v in X_train_vect:
    if v.size:
      X_train_vect_avg.append(v.mean(axis=0))
    else:
      X_train_vect_avg.append(np.zeros(100, dtype=float))
      
  probability = prediction = 'REAL' if loaded_model.predict(X_train_vect_avg) == 1 else 'FAKE'
  print(probability)

  #print(probability[0])
  return jsonify(result = probability)


if __name__=='__main__':
    app.run()

